In [1]:
from datetime import datetime
from pathlib import Path
from typing import List, NamedTuple, Tuple

import pandas as pd
import pybase64
import pydeck
from visionapi.sae_pb2 import PositionMessage
from visionapi.common_pb2 import MessageType, TypeMessage
from visionlib.saedump import DumpMeta, Event, message_splitter


class GPSPoint(NamedTuple):
    timestamp: datetime
    lat: float
    lon: float

    def to_coord_tuple(self) -> Tuple[float, float]:
        return (self.lat, self.lon)

class LineSegment(NamedTuple):
    start_lat: float
    start_lon: float
    end_lat: float
    end_lon: float

class Track(NamedTuple):
    source_file: Path
    gps_points: List[GPSPoint]

def parse_file_into_tracks(dump_file: Path, stream_filter: str = None) -> List[Track]:
    tracks = []
    track_points = []
    prev_timestamp = 0
    with open(dump_file, 'r') as f:
        message_iter = message_splitter(f)
        start_message = next(message_iter)
        dump_meta = DumpMeta.model_validate_json(start_message)

        for message in message_iter:
            event = Event.model_validate_json(message)
            if stream_filter and event.meta.source_stream != stream_filter:
                continue

            proto_bytes = pybase64.standard_b64decode(event.data_b64)

            position_msg = PositionMessage()
            position_msg.ParseFromString(proto_bytes)

            if not position_msg.fix:
                continue

            if position_msg.timestamp_utc_ms - prev_timestamp > 2000 and len(track_points) > 0:
                tracks.append(Track(source_file=dump_file, gps_points=track_points))
                track_points = []

            prev_timestamp = position_msg.timestamp_utc_ms

            track_points.append(GPSPoint(
                timestamp=datetime.fromtimestamp(position_msg.timestamp_utc_ms / 1000), 
                lat=position_msg.geo_coordinate.latitude, 
                lon=position_msg.geo_coordinate.longitude
            ))
    
    if len(track_points) > 0:
        tracks.append(Track(source_file=dump_file, gps_points=track_points))

    return tracks

def track_to_line_segments(track: Track) -> List[LineSegment]:
    segments = []
    for i in range(len(track.gps_points) - 1):
        segments.append(LineSegment(
            start_lat=track.gps_points[i].lat,
            start_lon=track.gps_points[i].lon, 
            end_lat=track.gps_points[i+1].lat,
            end_lon=track.gps_points[i+1].lon
        ))
    return segments

LOG_FILE = '/home/florian/workspaces/sae/starwit-awareness-engine/tools/sae-introspection/position-source-was-pods.saedump'

tracks: List[Track] = parse_file_into_tracks(LOG_FILE, stream_filter='positionsource:was-pod01')
track = tracks[1]

df_track = pd.DataFrame(track_to_line_segments(track))

line_layer = pydeck.Layer(
    'LineLayer',
    df_track,
    get_source_position='[start_lon, start_lat]',
    get_target_position='[end_lon, end_lat]',
    get_color='[255, 0, 0]',
    get_width=3,
)

ground_truth_scatter = pydeck.Layer(
    'ScatterplotLayer',
    df_track,
    get_position='[start_lon, start_lat]',
    get_color='[0, 0, 255, 128]',
    get_radius=0.24,
)

view_state = pydeck.data_utils.compute_view(df_track[['start_lon', 'start_lat']].values.tolist())
pydeck.Deck(
    layers=[
        ground_truth_scatter,
        line_layer,
    ], 
    initial_view_state=view_state,
    height=800,
).to_html(notebook_display=True, iframe_height=800)

In [7]:
pd.DataFrame(track_to_line_segments(track))

,start_lat,start_lon,end_lat,end_lon
0,52.427068,10.815812,52.427068,10.815768
1,52.427068,10.815768,52.427067,10.815727
2,52.427067,10.815727,52.427065,10.815683
3,52.427065,10.815683,52.427063,10.815598
4,52.427063,10.815598,52.427063,10.815555
...,...,...,...,...
647,52.427247,10.799253,52.427248,10.799245
648,52.427248,10.799245,52.427250,10.799238
649,52.427250,10.799238,52.427252,10.799227
650,52.427252,10.799227,52.427253,10.799223
